# Chapter 4: Tokenization

[Read this chapter online](https://jackluu.io/book/section-1-foundations/ch04-tokenization/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch04-tokenization.ipynb)

From *Teach Your Computer to Write: Build and Train an LLM from Zero* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 4: Tokenization

![The map with Tokens highlighted](https://jackluu.io/book/assets/diagrams/ch04-where-we-are.png)

*Figure 4.1: Tokenization in the big picture.*

In the last chapter we learned how to use tensors to store our information. So now that we're moving on to the next part of our map (Figure 4.1), we need to convert our text into numbers. A language model cannot understand patterns in our text if it's not converted to numbers. Neural networks operate on mathematics so we have to represent our text as numbers. This chapter will cover how to make our tokenizer which will take our text and split it into smaller chunks and give each one a number.

In this chapter you will:

- Understand why neural networks need numbers
- Build a character-level vocabulary from the Shakespeare dataset
- Write functions to encode text into integers and decode it back
- Prepare the data for training by splitting it into two sets

> **Words to Know**
>
> - **Token**: one small piece of text, here a single character.
> - **Tokenizer**: a translator that cuts text into tokens and gives each token a number.

## Theory

### Neural Networks Are Number Machines

![Each character maps to one ID](https://jackluu.io/book/assets/diagrams/ch04-text-to-ids.png)

*Figure 4.2: A tokenizer turns each character into a number.*

Neural networks aren't able to read. So how would you give them `a` string of characters such as "Hello"? They deal with arrays of numbers that they can multiply and add together. In order to give the neural network characters we have to turn them into numbers. This process is called tokenization (Figure 4.2).

You want to create some kind of dictionary that if the character `a` is the number `0` then it will always be `0`. This is known as your **vocabulary**.

### The Simplest Tokenizer

Text can be tokenized in many ways. You can have a dictionary for each word (tokenizing at the word level) or you can combine frequent characters like `th`.

We will be using the most basic form of tokenization which is at the character level. Every character that appears in our dataset will have its own corresponding integer.
Say our vocabulary was `{a: 0, b: 1, c: 2}`, then the word cab would be converted to `[2, 0, 1]`.

The Shakespeare data has 65 unique characters. It has all upper case characters, lower case characters, space, line break and some punctuation and one number. If we assign each of these characters an ID from 0-64 we can convert any sentence in shakespeare to a list of integers.

> **In Business**
>
> Tokenizing is something you need to keep in mind when building a text generator for your company's house style. Tokenizing at a character level will give you longer sequences of data. If you tokenize on a word level, you will have shorter sequences but your vocabulary becomes huge. Byte-Pair Encoding (BPE) is widely used in modern-day company assistants. It takes common groupings of characters and compresses them into one token. For simplicity we will be using character level, while our model learns from its archive (our Shakespeare dataset).

### The Training and Validation Split

![Splitting the dataset into training and validation sets](https://jackluu.io/book/assets/diagrams/ch04-train-val-split.png)

*Figure 4.3: Hiding part of the data lets us test whether the model learned.*

Once we encode all of Shakespeare into one long list of numbers, we split it into two piles (Figure 4.3):

- **Training set** (90%): The data the model studies to learn patterns.
- **Validation set** (10%): The data we hide from the model, used to test it later.

We withhold data so we can test whether our model has learned patterns that generalize to the rest of the text, or if it has simply memorized the passages we showed it during training. If a model does well on the training set but poorly on the validation set, we say it has memorized the training text. This is known as overfitting.

> **Watch Out**
>
> Our tokenizer knows nothing outside of the characters it saw during training. If you give it a number (`1` or `2` for example), an emoji, or a Chinese character, Python will throw a `KeyError` since that character isn't part of our 65-character vocabulary.

## Code

We write our tokenizer in Python, load the dataset, and encode it.

> **File**: `src/ch03_tokenizer.py`
> **Run it**: `python src/ch03_tokenizer.py`

```python
def build_vocab(text):
    # Find all unique characters in the text
    chars = sorted(set(text))

    # Dictionaries to translate between characters and their integer IDs
    char_to_id = {ch: i for i, ch in enumerate(chars)}
    id_to_char = {i: ch for i, ch in enumerate(chars)}

    return chars, char_to_id, id_to_char
```

The code does three things:

1. Line 3 uses `set(text)` to find every unique character in the entire text.
2. Line 3 uses `sorted()` to put them in alphabetical order so the IDs stay consistent.
3. Lines 6 and 7 build two lookup tables, which Python calls dictionaries: one to go from character to ID (`char_to_id`), and one to go back (`id_to_char`).

![Code flow for the tokenizer script](https://jackluu.io/book/assets/diagrams/ch04-code-flow.png)

*Figure 4.4: How the tokenizer processes the dataset.*

The script processes the dataset step by step (Figure 4.4). Next, we write the translation functions:

```python
def encode(text, char_to_id):
    # Convert a string into a list of integer IDs
    return [char_to_id[ch] for ch in text]

def decode(ids, id_to_char):
    # Convert a list of integer IDs back into a string
    return "".join([id_to_char[i] for i in ids])
```

Line 3 converts a string (a piece of text) into a list of whole-number IDs, and line 7 converts them back into text.

Now we run the script to see what our 65 characters look like and to test a quick round-trip.

```text
$ python src/ch03_tokenizer.py
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
...
--- Testing encode / decode ---
Original : 'Hello, World!'
Encoded  : [20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2]
Decoded  : 'Hello, World!'
Round-trip matches: True

--- Encoding the full dataset ---
```

**What just happened:**

- The script loaded over a million characters from the dataset.
- It found exactly 65 unique characters. Notice the space and newline characters at the start.
- It translated the string "Hello, World!" into a list of integers.
- It translated those integers back to prove no data was lost.

Finally, we encode the entire dataset into a PyTorch tensor and split it.

```text
Training tokens  : 1,003,854
Validation tokens: 111,540

Tokenizer ready! Ready for Chapter 5.
```

## Try It

See how different inputs are converted into numbers. We wrote a short script that imports our tokenizer and encodes a few business phrases.

> **Try It**
>
> Open the terminal and run the example script. Notice how every letter, space, and punctuation mark is assigned a specific number from our vocabulary.
>
> ```console title="Terminal"
> $ python src/examples/ch04_tokenizer_demo.py
> Demo: Encoding short business phrases
> 'invoice'       -> [47, 52, 60, 53, 47, 41, 43]
> 'URGENT!'       -> [33, 30, 19, 17, 26, 32, 2]
> 'Hello.'        -> [20, 43, 50, 50, 53, 8]
>
> ```

## Key Takeaways

- Neural networks work with numbers. They can't directly work with text. That's why we need to convert text into numbers first.
- This process is called tokenization. We tokenize our texts as numbers based on a static vocabulary.
- We will be using a character-level tokenizer with a vocabulary of 65 characters.
- `encode()` converts strings to lists of IDs. `decode()` converts IDs back to strings.
- We divide our data into training data ("Learn from these") and validation data ("Test on these").

## Check Your Understanding

1. How come we need to turn our text into numbers to use with a language model?
2. If we attempt to encode a character that wasn't in the training data using our character-level tokenizer, what will happen?
3. Why do we need a validation set?
4. How many unique characters are in our vocabulary?

## Assignment

> **Assignment 4: Same word, new numbers**
>
> **About 15 minutes. You change one word and write no new code.** Work in this chapter's Colab notebook or on your own laptop.
>
> 1. **Run it.** Run `src/examples/ch04_tokenizer_demo.py` unaltered. Copy the three rows it outputs, which will be phrases with their lists of numbers.
> 2. **Change one word.** Locate the line `"URGENT!",`. It's one of the three phrases we'll convert to numbers with this script. Replace the word `URGENT` with `urgent`, making it all lowercase letters, but leave the rest of the line intact, so it says `"urgent!",`. Run the script again.
> 3. **Compare.** Which numbers changed in the second row? Which did not, and what letter does it represent? Two numbers in your new row are also in the `invoice` row. What two letters do they represent?
> 4. **Explain it.** In two or three sentences someone who manages you would understand, explain why the model sees `URGENT!` and `urgent!` are two different pieces of text when people reading them understand it's the same word.
>
> **Hand in:** both sets of three rows, your answers to step 3, and your sentences.

## Further Reading

**Why real models do not use a character vocabulary.** Chapter 4 assigns a unique token to every character. This keeps the vocabulary small and the code concise. Models used in production tokenize text into word pieces instead. Common words are not split, rare words are split into subwords, and no word is ever unknown. This paper describes the algorithm, which remains the basis of the tokenizers included with modern models today.


Sennrich, R., Haddow, B., & Birch, A. (2015). *Neural machine translation of rare words with subword units* (arXiv:1508.07909). arXiv. https://doi.org/10.48550/arXiv.1508.07909

---

### `src/ch03_tokenizer.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch03_tokenizer.py"   # a cell has none, and the file uses it to find the text

"""
Build a character-level tokenizer.
This file belongs to Chapter 4.
Run: python src/ch03_tokenizer.py
"""
import os
import sys
import torch

# Settings
DATA_PATH = os.path.join(os.path.dirname(__file__), "data", "shakespeare.txt")

# --- The Idea ---

def build_vocab(text):
    # Find all unique characters in the text
    chars = sorted(set(text))

    # Dictionaries to translate between characters and their integer IDs
    char_to_id = {ch: i for i, ch in enumerate(chars)}
    id_to_char = {i: ch for i, ch in enumerate(chars)}

    return chars, char_to_id, id_to_char

def encode(text, char_to_id):
    # Convert a string into a list of integer IDs
    return [char_to_id[ch] for ch in text]

def decode(ids, id_to_char):
    # Convert a list of integer IDs back into a string
    return "".join([id_to_char[i] for i in ids])

# --- Demo ---
if __name__ == "__main__":
    print("Chapter 4: Tokenization\n")

    if not os.path.exists(DATA_PATH):
        print("ERROR: shakespeare.txt not found.")
        print("Please run:  python src/utils/download_data.py")
        sys.exit(1)

    with open(DATA_PATH, "r", encoding="utf-8") as f:
        text = f.read()

    print(f"Loaded {len(text):,} characters from shakespeare.txt")
    print(f"First 100 chars: {repr(text[:100])}")

    print("\n--- Building the vocabulary ---")
    chars, char_to_id, id_to_char = build_vocab(text)
    vocab_size = len(chars)
    print(f"Unique characters ({vocab_size} total):\n  " + "".join(chars))

    print("\n--- Testing encode / decode ---")
    sample = "Hello, World!"
    encoded = encode(sample, char_to_id)
    decoded = decode(encoded, id_to_char)

    print(f"Original : {repr(sample)}")
    print(f"Encoded  : {encoded}")
    print(f"Decoded  : {repr(decoded)}")
    print(f"Round-trip matches: {sample == decoded}")

    print("\n--- Encoding the full dataset ---")
    all_ids = encode(text, char_to_id)

    # Store IDs in a PyTorch tensor for efficient model training
    data = torch.tensor(all_ids, dtype=torch.long)

    print(f"Full dataset as tensor: shape={data.shape}, dtype={data.dtype}")
    print(f"First 20 token IDs: {data[:20].tolist()}")
    print(f"Decoded back: {repr(decode(data[:20].tolist(), id_to_char))}")

    print("\n--- Splitting into train and validation sets ---")
    # Use 90% for training and 10% for validation to evaluate performance
    split = int(0.9 * len(data))
    train_data = data[:split]
    val_data   = data[split:]

    print(f"Training tokens  : {len(train_data):,}")
    print(f"Validation tokens: {len(val_data):,}")

    print("\nTokenizer ready! Ready for Chapter 5.")

---

### `src/examples/ch04_tokenizer_demo.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch04_tokenizer_demo.py"   # a cell has none, and the file uses it to find the text

"""Shows how different text inputs map to token IDs."""
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", ".."))

from src.ch03_tokenizer import build_vocab, encode

DATA_PATH = os.path.join(os.path.dirname(__file__), "..", "data", "shakespeare.txt")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()

chars, char_to_id, _ = build_vocab(text)

print("Demo: Encoding short business phrases")
phrases = [
    "invoice",
    "URGENT!",
    "Hello."
]

for p in phrases:
    # Notice how spaces and punctuation get their own IDs
    print(f"{repr(p):<15} -> {encode(p, char_to_id)}")